# QCodeDiscovery.jl — usage demonstration

A **pure-Julia, GPU-accelerated** package for catalog-blind discovery of bivariate-bicycle (BB)
quantum LDPC codes — the Julia migration of the reproduction of *arXiv:2606.02418*.
No C/C++ dependencies: the HiGHS MILP, the `ldpc` BP-OSD decoder, and the `igraph` BLISS
canonicalizer are all reimplemented natively in Julia.

> Run from the repo: `jupyter notebook julia/examples/demo.ipynb` (needs IJulia + the `julia/` project instantiated).

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # activate the QCodeDiscovery.jl package (julia/)
using QCodeDiscovery

## 1. Construct a code and verify it (the kernel)

`BBCode(l, m, A, B)` builds `H_X=(A|B)`, `H_Z=(B^T|A^T)` over `R = F2[x,y]/(xˡ−1, yᵐ−1)`.

In [ ]:
gross = BBCode(12, 6, "y+y^2+x^3", "y^3+x+x^2")   # the gross code [[144,12,12]]
(n = gross.n, k = css_k(gross), stab_weight = stabilizer_weight(gross), css_valid = css_valid(gross))

## 2. Exact distance — pure-Julia Brouwer–Zimmermann (replaces the HiGHS C++ MILP)

`min_distance_bz` returns a **certified** minimum distance with no external solver.

In [ ]:
min_distance_bz(BBCode(6, 6, "x^3+y+y^2", "y^3+x+x^2"))   # [[72,12,6]] -> d=6, certified=true

## 3. Theorem witnesses

`thm:ab_d2` (A=B ⇒ d=2) and `lem:crt_k` (univariate ⇒ k=8ℓ/3).

In [ ]:
verify_ab_d2(6, 6, "1+x+y"), verify_crt_k(6, 6)

## 4. Blind discovery (catalog-blind) → post-hoc validation

The search uses naive random seeds and FOM fitness — never the paper's codes. Validation is the
only step that consults the reference landmarks, and only afterwards.

In [ ]:
out = blind_search_css([(6, 6)]; n_random=300, distance_budget=6, distance_method=:bposd, seed=0)
out.archive_elites

In [ ]:
validate(out.archive_elites; kind=:css)

## 5. Structure — `dedup` proves the blind [[144,12,12]] IS the gross code

A canonical hash of the colored Tanner graph (pure-Julia individualization-refinement, replacing
igraph/BLISS). A blindly-found code with different polynomials hashes equal to the gross code.

In [ ]:
blind = BBCode(12, 6, "x^9y^5+x^10y^2+x^11y^2", "x^2y^4+x^2y^5+x^5")  # found blind earlier
canonical_hash(blind.HX, blind.HZ) == canonical_hash(gross.HX, gross.HZ)   # true: same code, relabeled

## 6. BP-OSD decoder (replaces ldpc C++) and GPU / parallel

`bposd_distance` is a native belief-propagation + OSD decoder (stochastic upper bound).
`batched_rank` runs GF(2) rank across CPU threads — and on the A100 GPU when `CUDA.jl` is loaded.

In [ ]:
bposd_distance(gross; trials=200, seed=0)   # d_bound = 12

In [ ]:
batched_rank([gross.HX, gross.HZ]), cuda_available()

---
That is the full pipeline — construction, certified exact distance, theorems, blind discovery,
validation, structural dedup, decoding, and GPU/parallel execution — all in pure Julia.
See `julia/README.md` and `julia/bin/qcode-discover.jl` (CLI).